In [1]:
from mava.networks.retention import MultiScaleRetention
from omegaconf import DictConfig
import jax
import jax.numpy as jnp
import copy

# jax.config.update("jax_enable_x64", True)

bsz = 16
num_agents = 4
obs_dim = 11
num_time_steps = 400
seq_len = num_agents * num_time_steps

retnet_embed_dim = 32
retnet_num_heads = 2

2025-02-26 11:25:25.137788: W external/xla/xla/service/gpu/nvptx_compiler.cc:765] The NVIDIA driver's CUDA version is 12.4 which is older than the ptxas CUDA version (12.8.61). Because the driver is older than the ptxas version, XLA is disabling parallel compilation, which may slow down compilation. You should update your NVIDIA driver or use the NVIDIA-provided CUDA forward compatibility packages.
/home/ruanjohn/miniconda3/envs/mava/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
memory_config = DictConfig(
    {
        "type": "rec_sable",
        "decay_scaling_factor": 0.3,
        "timestep_positional_encoding": True,
        "timestep_chunk_size": None,
    }
)

decay_kappas = 1 - jnp.exp(jnp.linspace(jnp.log(1 / 32), jnp.log(1 / 512), retnet_num_heads))
decay_kappas = jnp.log(decay_kappas * memory_config.decay_scaling_factor)
decay_kappas = decay_kappas[None, :, None, None]

In [3]:
msr = MultiScaleRetention(
    embed_dim=retnet_embed_dim,
    n_head=retnet_num_heads,
    n_agents=num_agents,
    memory_config=memory_config,
    masked=False,
    decay_scaling_factor=0.3
)

In [4]:
key = jax.random.PRNGKey(0)
key, subkey = jax.random.split(key)

obs = jax.random.normal(subkey, (bsz, seq_len, retnet_embed_dim))

# assuming no resets
dones = jnp.zeros((bsz, seq_len), dtype=bool)

init_hstate = jnp.zeros((bsz, retnet_num_heads, retnet_embed_dim//retnet_num_heads, retnet_embed_dim//retnet_num_heads))
init_scale = jnp.ones((bsz, retnet_num_heads, 1, 1))
step_counts = jnp.arange(num_time_steps)
step_counts = step_counts[None, ...].repeat(bsz, axis=0)[..., None].repeat(num_agents, axis=-1)
step_counts = step_counts.reshape(bsz, seq_len)

In [5]:
key, init_key = jax.random.split(key)
params = msr.init(
    init_key,
    obs,
    obs,
    obs,
    init_hstate,
    dones,
    step_counts,
    1,
    init_scale,
)

In [6]:
hstate = copy.deepcopy(init_hstate)
scale = copy.deepcopy(init_scale)
act_output = []


# for the decoder we use the chunkwise
for step in range(num_time_steps):

    # todo: reset later

    updated_scale = scale * jnp.exp(decay_kappas) + 1
    scale_factor = jnp.sqrt(scale) * jnp.exp(decay_kappas) / jnp.sqrt(updated_scale)
    hstate = hstate * scale_factor
    scale = updated_scale

    obs_i = obs[:, step*num_agents:(step+1)*num_agents, ...]
    dones_i = dones[:, step*num_agents:(step+1)*num_agents]
    step_counts_i = step_counts[:, step*num_agents:(step+1)*num_agents]

    out, hstate = msr.apply(params, obs_i, obs_i, obs_i, hstate, dones_i, step_counts_i, 1, scale)
    act_output.append(out)

In [7]:
act_output = jnp.concatenate(act_output, axis=1)

In [8]:
act_output.shape

(16, 1600, 32)

In [9]:
hstate = copy.deepcopy(init_hstate)
scale = copy.deepcopy(init_scale)
train_out, _ = msr.apply(params, obs, obs, obs, hstate, dones, step_counts, 1, scale)

In [10]:
train_out.shape

(16, 1600, 32)

In [11]:
total_error = jnp.mean(jnp.abs(train_out - act_output))
total_error

Array(0.0041851, dtype=float32)

In [ ]:
jnp.abs(train_out - act_output)

Array([[[2.15191394e-05, 1.64760277e-05, 2.13794410e-05, ...,
         5.45568764e-06, 3.58596444e-05, 1.95093453e-05],
        [8.00751150e-06, 4.12482768e-05, 7.86799937e-05, ...,
         5.77922910e-05, 7.47246668e-06, 5.75035810e-05],
        [4.33064997e-06, 1.48778781e-07, 9.23499465e-06, ...,
         1.36811286e-05, 2.91690230e-06, 1.88499689e-06],
        ...,
        [9.48642846e-04, 9.16558318e-03, 6.46043615e-03, ...,
         8.32449645e-04, 1.51597019e-02, 4.53709438e-03],
        [4.94202599e-04, 1.08194090e-02, 4.91908239e-03, ...,
         6.42593019e-03, 5.62446937e-03, 4.14799154e-03],
        [4.39843163e-04, 4.59900312e-03, 2.81055458e-03, ...,
         7.89343752e-03, 6.31101429e-04, 4.66275029e-03]],

       [[1.73691660e-06, 1.24159269e-05, 5.47524542e-06, ...,
         1.81477517e-05, 1.49710104e-05, 4.22634184e-06],
        [2.62353569e-06, 4.34154645e-05, 9.51578841e-07, ...,
         1.18520111e-05, 2.39275396e-05, 2.39722431e-05],
        [6.67944551e-06, 

: 